# League of Legends object detection: fair three-model comparison

**What this notebook does**

1. Connects to Google Drive and copies the dataset onto Colab's fast local disk.
2. **Part A:** scores the three models you already trained in April on the *test* split (no training needed).
3. **Part B:** trains every model in the `MODELS` list with identical settings, one after another.
4. **Part C:** scores the new models on *val* and *test* and saves tidy tables for the report.

**How to run:** Runtime > Change runtime type > pick a GPU, then Runtime > Run all.
If Colab disconnects, just run all again. Finished models are skipped and a half-trained one continues where it stopped.

## 1. Setup

We pack the tool we need `ultralytics`, the library that contains the YOLO models, connect to Google Drive, and check that the GPU is switched on.

In [2]:
# Install the YOLO library. "-q" means quiet, so it does not fill the screen with text.
!pip install -q ultralytics

from google.colab import drive
drive.mount("/content/drive")   # A popup asks for permission to read your Drive.

import torch
# cuda = the GPU. If this prints False, training still works but is roughly 10 to 20 times slower.
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Safety stop: training on the CPU works but takes many hours, so we refuse to start by accident.
# If you really want CPU training, change ALLOW_CPU to True.
ALLOW_CPU = False
assert torch.cuda.is_available() or ALLOW_CPU, (
    "No GPU found. Go to Runtime > Change runtime type, choose a GPU (for example T4), then Run all again."
)

Mounted at /content/drive
GPU available: True
GPU name: Tesla T4


## 2. Settings

Everything you might want to change lives in this one cell. The golden rule of a fair comparison: **only the model changes, everything else stays the same.**

In [3]:
from pathlib import Path

# ---- The models to compare. Add or remove lines here to train more or fewer models. ----
MODELS = [
    "yolov8n.pt",
    "yolo11n.pt",
    "yolo26n.pt",
]

# ---- Identical training settings for every model ----
EPOCHS   = 30   # Maximum number of passes through the training images.
PATIENCE = 30    # Stop early if validation has not improved for this many epochs (saves time and electricity).
IMGSZ    = 640   # Images are resized to 640x640 before going into the model.
BATCH    = 16    # How many images the model looks at before each small correction.
SEED     = 0     # Fixes the "random" choices so a rerun gives the same result.

# ---- Class names. The order must match the numbers inside your label files. ----
CLASS_NAMES = ["ally_minion", "enemy_minion", "enemy_champion", "ally_champion", "tower", "player"]

# ---- Expected image counts, used as a safety check after copying ----
EXPECTED = {"train": 661, "val": 94, "test": 79}

# ---- Where the project lives on Drive. Leave as None to let the notebook find it. ----
DRIVE_PROJECT = None   # example: Path("/content/drive/MyDrive/Uni/computer_vision")

# ---- The April runs (30 epochs each) that Part A will score on the test split ----
OLD_RUNS = {
    "yolov8n (April, 30 ep)": "lol_detect_yolov8n",
    "yolo11n (April, 30 ep)": "lol_detect_baseline3",
    "yolo26n (April, 30 ep)": "lol_detect_yolo26n",
}

In [4]:
MYDRIVE = Path("/content/drive/MyDrive")

def find_project() -> Path:
    # Look for a folder called computer_vision that contains the dataset.
    # We only look up to three folders deep, because searching all of Drive is very slow.
    for pattern in ("computer_vision", "*/computer_vision", "*/*/computer_vision", "*/*/*/computer_vision"):
        for candidate in MYDRIVE.glob(pattern):
            if (candidate / "screenshot_generator" / "dataset" / "images" / "train").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find the computer_vision folder. Set DRIVE_PROJECT by hand in the settings cell."
    )

if DRIVE_PROJECT is None:
    DRIVE_PROJECT = find_project()

DRIVE_DATASET = DRIVE_PROJECT / "screenshot_generator" / "dataset"
OLD_RUNS_DIR  = DRIVE_PROJECT / "runs"       # the April results (we only read from here)
NEW_RUNS_DIR  = DRIVE_PROJECT / "runs_v2"    # everything this notebook creates goes here
SUMMARY_DIR   = NEW_RUNS_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# A short label for the current settings. It goes into run names and summary file names,
# so each experiment keeps its own results and nothing gets overwritten.
SETTINGS_TAG = f"e{EPOCHS}_p{PATIENCE}_img{IMGSZ}"

print("Project :", DRIVE_PROJECT)
print("Dataset :", DRIVE_DATASET)
print("New runs:", NEW_RUNS_DIR)
print("Settings:", SETTINGS_TAG)

Project : /content/drive/MyDrive/Projects/Uni Stuff/computer_vision
Dataset : /content/drive/MyDrive/Projects/Uni Stuff/computer_vision/screenshot_generator/dataset
New runs: /content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2
Settings: e30_p30_img640


## 3. Copy the dataset to Colab's local disk

Reading images straight from Drive is like walking to the library for every single page of a book. Copying the dataset once is like borrowing the whole book and reading it at your desk. Training then runs much faster.

**What can go wrong:** a half-finished copy would silently train on fewer images. The count check at the end catches that.

In [5]:
import shutil

LOCAL_DATASET = Path("/content/dataset")
IMAGE_TYPES = {".png", ".jpg", ".jpeg"}

def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for f in folder.iterdir() if f.suffix.lower() in IMAGE_TYPES)

for split in EXPECTED:
    for kind in ("images", "labels"):
        src = DRIVE_DATASET / kind / split
        dst = LOCAL_DATASET / kind / split
        # Skip the copy if this split is already complete (for example after "Run all" a second time).
        if kind == "images" and count_images(dst) == EXPECTED[split]:
            continue
        print(f"Copying {kind}/{split} ...")
        shutil.copytree(src, dst, dirs_exist_ok=True)

# Safety check: do we have exactly the number of images we expect, and one label file per image?
for split, expected in EXPECTED.items():
    n_img = count_images(LOCAL_DATASET / "images" / split)
    n_lbl = len(list((LOCAL_DATASET / "labels" / split).glob("*.txt")))
    status = "OK" if n_img == expected else "MISMATCH"
    print(f"{split:5s}: {n_img} images, {n_lbl} label files (expected {expected} images) -> {status}")
    assert n_img == expected, f"{split} has {n_img} images but {expected} were expected."

Copying images/train ...
Copying labels/train ...
Copying images/val ...
Copying labels/val ...
Copying images/test ...
Copying labels/test ...
train: 661 images, 661 label files (expected 661 images) -> OK
val  : 94 images, 94 label files (expected 94 images) -> OK
test : 79 images, 79 label files (expected 79 images) -> OK


In [6]:
import yaml

# The dataset "address card" that tells YOLO where the images are and what the classes are called.
# We write a fresh one because the old dataset.yaml points to a folder on your Mac.
DATA_YAML = LOCAL_DATASET / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(LOCAL_DATASET),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}, sort_keys=False))
print(DATA_YAML.read_text())

path: /content/dataset
train: images/train
val: images/val
test: images/test
names:
  0: ally_minion
  1: enemy_minion
  2: enemy_champion
  3: ally_champion
  4: tower
  5: player



## 4. A helper that scores a model

We will score many models on two splits, so we write the recipe once and reuse it.

Quick reminder of the four numbers, using a game of "spot the minion":

- **Precision:** of everything the model shouted "minion!" at, how many really were minions? (Low precision = lots of false alarms.)
- **Recall:** of all the real minions on screen, how many did the model find? (Low recall = it misses things.)
- **mAP@50:** one overall score where a box counts as correct if it overlaps the true box by at least half.
- **mAP@50:95:** a stricter version that also rewards boxes that fit *tightly*.

In [7]:
import pandas as pd
from ultralytics import YOLO

def evaluate(weights: Path, label: str, split: str, save_name: str):
    # Scores one model on one split. Returns (overall row, per-class rows).
    model = YOLO(str(weights))
    metrics = model.val(
        data=str(DATA_YAML),
        split=split,                 # "val" or "test"
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(NEW_RUNS_DIR / "eval"),
        name=save_name,
        exist_ok=True,               # overwrite the folder instead of creating eval2, eval3, ...
        plots=True,                  # saves the confusion matrix and PR curves for the report
        verbose=False,
    )
    box = metrics.box
    n_params = sum(p.numel() for p in model.model.parameters())

    overall = {
        "model": label,
        "split": split,
        "precision": round(float(box.mp), 3),
        "recall": round(float(box.mr), 3),
        "mAP50": round(float(box.map50), 3),
        "mAP50-95": round(float(box.map), 3),
        "inference_ms_per_image": round(float(metrics.speed["inference"]), 2),
        "params_millions": round(n_params / 1e6, 2),
    }

    per_class = []
    # ap_class_index lists the classes that actually appear in this split.
    for i, class_id in enumerate(box.ap_class_index):
        per_class.append({
            "model": label,
            "split": split,
            "class": metrics.names[int(class_id)],
            "precision": round(float(box.p[i]), 3),
            "recall": round(float(box.r[i]), 3),
            "AP50": round(float(box.ap50[i]), 3),
            "AP50-95": round(float(box.ap[i]), 3),
        })
    return overall, per_class

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 5. Part A: score the April models on the test split

These models already exist, so this takes about a minute and costs almost nothing. It gives you honest final numbers for the work you already did, because the test games were never used to make any decision during training.

In [8]:
old_overall, old_per_class = [], []

for label, folder in OLD_RUNS.items():
    weights = OLD_RUNS_DIR / folder / "weights" / "best.pt"
    if not weights.exists():
        print(f"Skipping {label}: no weights at {weights}")
        continue
    for split in ("val", "test"):
        overall, per_class = evaluate(weights, label, split, save_name=f"april_{folder}_{split}")
        old_overall.append(overall)
        old_per_class.extend(per_class)

old_overall_df = pd.DataFrame(old_overall)
old_per_class_df = pd.DataFrame(old_per_class)
old_overall_df.to_csv(SUMMARY_DIR / "april_models_overall.csv", index=False)
old_per_class_df.to_csv(SUMMARY_DIR / "april_models_per_class.csv", index=False)
old_overall_df

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3251.9±1969.1 MB/s, size: 2176.4 KB)
val: Scanning /content/dataset/labels/val... 94 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 94/94 289.3it/s 0.3s
val: New cache created: /content/dataset/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2it/s 5.0s
                   all         94        431      0.872      0.905      0.928      0.686
Speed: 1.2ms preprocess, 3.7ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to /content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2/eval/april_lol_detect_yolov8n_val
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 

,model,split,precision,recall,mAP50,mAP50-95,inference_ms_per_image,params_millions
0,"yolov8n (April, 30 ep)",val,0.872,0.905,0.928,0.686,3.67,3.01
1,"yolov8n (April, 30 ep)",test,0.880,0.848,0.921,0.668,3.22,3.01
2,"yolo11n (April, 30 ep)",val,0.851,0.907,0.923,0.675,3.12,2.59
3,"yolo11n (April, 30 ep)",test,0.877,0.874,0.914,0.668,3.37,2.59
4,"yolo26n (April, 30 ep)",val,0.826,0.911,0.921,0.681,3.22,2.51
5,"yolo26n (April, 30 ep)",test,0.851,0.860,0.897,0.663,3.58,2.51


## 6. Part B: train every model in the list

The loop below is like baking three cakes with the same oven temperature and the same timer, changing only the recipe.

Three safety features:

- **Early stopping (`patience`):** if the model stops improving, training ends instead of burning GPU time for nothing.
- **Skip finished models:** a small `DONE.txt` note is left in each finished run folder.
- **Resume:** if Colab disconnects in the middle, the model continues from its last saved checkpoint.

**What can go wrong:** if you change a setting (for example `EPOCHS`) after some models are finished, the finished ones keep the old setting and the comparison is no longer fair. The run names include the settings, so a changed setting automatically starts a fresh set of runs.

In [9]:
def run_name(model_file: str) -> str:
    # Example: "detect_yolo11n_e100_p20_img640"
    return f"detect_{Path(model_file).stem}_{SETTINGS_TAG}"

for model_file in MODELS:
    name = run_name(model_file)
    run_dir = NEW_RUNS_DIR / name
    done_marker = run_dir / "DONE.txt"
    last_checkpoint = run_dir / "weights" / "last.pt"

    print("=" * 70)
    if done_marker.exists():
        print(f"{name}: already finished, skipping.")
        continue

    if last_checkpoint.exists():
        print(f"{name}: found a checkpoint, resuming.")
        try:
            # device is passed again on purpose: without it, a run that started on the CPU would stay on the CPU.
            YOLO(str(last_checkpoint)).train(resume=True, device=0 if torch.cuda.is_available() else "cpu")
        except AssertionError as err:
            # Ultralytics raises this when the run had in fact finished before the disconnect.
            print(f"Nothing left to resume ({err}). Marking as finished.")
    else:
        print(f"{name}: starting training.")
        YOLO(model_file).train(
            data=str(DATA_YAML),
            epochs=EPOCHS,
            patience=PATIENCE,
            imgsz=IMGSZ,
            batch=BATCH,
            seed=SEED,
            device=0 if torch.cuda.is_available() else "cpu",
            project=str(NEW_RUNS_DIR),   # saved on Drive, so nothing is lost if Colab disconnects
            name=name,
            exist_ok=True,
            plots=True,
        )

    done_marker.write_text("finished")

print("All models trained.")

detect_yolov8n_e30_p30_img640: starting training.
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=detect_yolov

## 7. Part C: score the new models on val and test

**Important habit:** look at the test numbers once, at the end. If you keep tweaking settings until the test score looks good, the test set quietly turns into a second validation set and stops being an honest exam.

In [10]:
new_overall, new_per_class = [], []

for model_file in MODELS:
    name = run_name(model_file)
    run_dir = NEW_RUNS_DIR / name
    weights = run_dir / "weights" / "best.pt"
    if not weights.exists():
        print(f"Skipping {name}: not trained yet.")
        continue

    # results.csv has one row per epoch, so its length tells us when early stopping kicked in.
    history = pd.read_csv(run_dir / "results.csv")
    history.columns = [c.strip() for c in history.columns]
    epochs_run = len(history)
    best_epoch = int(history.loc[history["metrics/mAP50-95(B)"].idxmax(), "epoch"])

    for split in ("val", "test"):
        overall, per_class = evaluate(weights, Path(model_file).stem, split, save_name=f"{name}_{split}")
        overall["settings"] = SETTINGS_TAG
        overall["epochs_run"] = epochs_run
        overall["best_epoch"] = best_epoch
        new_overall.append(overall)
        new_per_class.extend(per_class)

new_overall_df = pd.DataFrame(new_overall)
new_per_class_df = pd.DataFrame(new_per_class)
if not new_per_class_df.empty:
    new_per_class_df.insert(2, "settings", SETTINGS_TAG)
new_overall_df.to_csv(SUMMARY_DIR / f"new_models_overall_{SETTINGS_TAG}.csv", index=False)
new_per_class_df.to_csv(SUMMARY_DIR / f"new_models_per_class_{SETTINGS_TAG}.csv", index=False)
new_overall_df

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,922 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5281.3±593.4 MB/s, size: 2211.0 KB)
val: Scanning /content/dataset/labels/val.cache... 94 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 94/94 32.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1it/s 5.6s
                   all         94        431      0.883       0.91      0.929      0.689
Speed: 2.4ms preprocess, 3.4ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to /content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2/eval/detect_yolov8n_e30_p30_img640_val
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,922 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 m

,model,split,precision,recall,mAP50,mAP50-95,inference_ms_per_image,params_millions,settings,epochs_run,best_epoch
0,yolov8n,val,0.883,0.910,0.929,0.689,3.42,3.14,e30_p30_img640,30,30
1,yolov8n,test,0.891,0.869,0.923,0.675,3.38,3.14,e30_p30_img640,30,30
2,yolo11n,val,0.867,0.893,0.929,0.685,3.37,2.61,e30_p30_img640,30,30
3,yolo11n,test,0.888,0.843,0.914,0.672,4.22,2.61,e30_p30_img640,30,30
4,yolo26n,val,0.890,0.871,0.924,0.681,3.62,2.54,e30_p30_img640,30,25
5,yolo26n,test,0.880,0.854,0.901,0.667,3.68,2.54,e30_p30_img640,30,25


In [11]:
# Per-class test results side by side: one row per class, one column per model.
# This is the table that shows *where* each model is strong or weak.
if not new_per_class_df.empty:
    test_only = new_per_class_df[new_per_class_df["split"] == "test"]
    display(test_only.pivot(index="class", columns="model", values="AP50-95"))

model,yolo11n,yolo26n,yolov8n
class,,,
ally_champion,0.712,0.738,0.736
ally_minion,0.566,0.583,0.559
enemy_champion,0.735,0.697,0.736
enemy_minion,0.484,0.529,0.523
player,0.815,0.801,0.787
tower,0.718,0.654,0.711


## 8. Where everything was saved

All of this is on Google Drive inside `computer_vision/runs_v2`:

- `summary/` : the CSV tables (overall and per class). The April tables have fixed names; the new-model tables end in the settings tag, for example `new_models_overall_e30_p30_img640.csv`
- `detect_<model>_.../` : training curves (`results.png`), weights and training batches for each new model
- `eval/<run>_test/` : confusion matrix and PR curves on the test split, ready to use as report figures

In [12]:
for f in sorted(SUMMARY_DIR.glob("*.csv")):
    print(f)

/content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2/summary/april_models_overall.csv
/content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2/summary/april_models_per_class.csv
/content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2/summary/new_models_overall_e100_p20_img640.csv
/content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2/summary/new_models_overall_e30_p30_img640.csv
/content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2/summary/new_models_per_class_e100_p20_img640.csv
/content/drive/MyDrive/Projects/Uni Stuff/computer_vision/runs_v2/summary/new_models_per_class_e30_p30_img640.csv
